 This notebooks imports the daily updated S&P_500_all_assets.csv from Kaggle.

 Link: https://www.kaggle.com/datasets/yash16jr/s-and-p500-daily-update-dataset$0

In [0]:
%pip install kagglehub[pandas-datasets]


In [0]:
import os

try:
    # Holt den Key sicher aus dem Job-Parameter (Widget)
    kaggle_token = dbutils.widgets.get("KAGGLE_API_TOKEN")
    
    # Setzt ihn als Umgebungsvariable für die Kaggle-Bibliothek
    os.environ["KAGGLE_API_TOKEN"] = kaggle_token
    
    print("Kaggle API Token erfolgreich und sicher aus den Job-Parametern geladen!")
except Exception as e:
    print("Fehler: Konnte den Parameter 'KAGGLE_API_TOKEN' nicht finden. "
          "Haben Sie ihn in den Job-Details hinterlegt?")



In [0]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df_pd = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "yash16jr/s-and-p500-daily-update-dataset",
    "SnP_daily_update.csv",
)

print("Pandas DF Shape:", df_pd.shape)
df_pd.head()


In [0]:
import pandas as pd

# Ausgangspunkt: df_pd aus Kaggle
df = df_pd.copy()

# 1) Ticker-Zeile extrahieren (Zeile 0)
header_row = df.iloc[0]

# 2) Nur die Datenzeilen behalten (Zeile 0 = "Ticker", Zeile 1 = "Date")
df_data = df[~df["Price"].isin(["Ticker", "Date"])].copy()

# 3) Hilfsfunktion: relevante Spalten je Kennzahl finden
def get_cols(prefix: str):
    return [c for c in df.columns if c == prefix or c.startswith(prefix + ".")]

open_cols   = get_cols("Open")
high_cols   = get_cols("High")
low_cols    = get_cols("Low")
close_cols  = get_cols("Close")
volume_cols = get_cols("Volume")

# 4) Hilfsfunktion: eine Kennzahl in Long-Format bringen
def melt_metric(metric_name: str, metric_cols):
    # neutraler value_name, um Kollision mit bestehenden Spalten zu vermeiden
    tmp = df_data.melt(
        id_vars=["Price"],
        value_vars=metric_cols,
        var_name="Column",
        value_name=f"{metric_name}_val"
    )
    # Ticker anhand der ersten Zeile zuordnen
    tmp["Ticker"] = tmp["Column"].map(lambda col: header_row[col])
    # leere Werte und Spalten ohne Ticker verwerfen
    tmp = tmp.dropna(subset=[f"{metric_name}_val", "Ticker"])
    # Kennzahl-Spalte auf den echten Namen zurückbenennen
    tmp = tmp.rename(columns={f"{metric_name}_val": metric_name})
    return tmp[["Price", "Ticker", metric_name]]

# 5) OHLCV unpivoten
open_m   = melt_metric("Open",   open_cols)
high_m   = melt_metric("High",   high_cols)
low_m    = melt_metric("Low",    low_cols)
close_m  = melt_metric("Close",  close_cols)
volume_m = melt_metric("Volume", volume_cols)

# 6) Alles auf (Price, Ticker) mergen
df_tidy = close_m.merge(open_m,   on=["Price", "Ticker"], how="left")
df_tidy = df_tidy.merge(high_m,   on=["Price", "Ticker"], how="left")
df_tidy = df_tidy.merge(low_m,    on=["Price", "Ticker"], how="left")
df_tidy = df_tidy.merge(volume_m, on=["Price", "Ticker"], how="left")

# 7) Datentypen und Datumsspalte setzen
df_tidy["Date"] = pd.to_datetime(df_tidy["Price"], errors="coerce").dt.date

for col in ["Open", "High", "Low", "Close", "Volume"]:
    df_tidy[col] = pd.to_numeric(df_tidy[col], errors="coerce")

df_tidy = (
    df_tidy
    .drop(columns=["Price"])
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

df_tidy.head()


In [0]:
df_spark = spark.createDataFrame(df_tidy)
df_spark.printSchema()
df_spark.show(5)


In [0]:
(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")  # bei Bedarf in "append" ändern
    .saveAsTable("thekitchen.kg.sp500_aa_daily")
)


In [0]:
%skip
df_check = spark.table("thekitchen.kg.sp500_aa_daily")
df_check.show(10)
